In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import duckdb
from src.data.ingestion.statcast_client import project_root
from src.features.plate_discipline import swing_count_sql, whiff_count_sql, swing_sql

con = duckdb.connect()
glob_path = str(project_root() / "data" / "raw" / "statcast_*.parquet")
con.execute(f"CREATE OR REPLACE VIEW pitches AS SELECT * FROM read_parquet('{glob_path}')")

print(swing_sql())
print(swing_count_sql())

description IN ('foul', 'foul_tip', 'hit_into_play', 'swinging_strike', 'swinging_strike_blocked')
SUM(CASE WHEN description IN ('foul', 'foul_tip', 'hit_into_play', 'swinging_strike', 'swinging_strike_blocked') THEN 1 ELSE 0 END) AS swings


In [2]:
q = f"""
SELECT
    pitch_type,
    COUNT(*)                      AS pitches,
    {swing_count_sql()},
    {whiff_count_sql()},
    ROUND(AVG(release_speed), 1)  AS avg_velo
FROM pitches
WHERE pitch_type IS NOT NULL
GROUP BY pitch_type
HAVING swings >= 100
ORDER BY whiffs * 1.0 / swings DESC
"""
con.execute(q).df()

,pitch_type,pitches,swings,whiffs,avg_velo
0,FS,385,201.0,68.0,86.4
1,SL,1407,673.0,212.0,86.0
2,CH,1084,541.0,163.0,85.5
3,ST,666,302.0,81.0,82.1
4,CU,765,332.0,77.0,79.5
5,FC,909,436.0,82.0,89.0
6,FF,3541,1700.0,298.0,94.1
7,SI,1605,742.0,74.0,93.0


In [3]:
q = f"""
WITH pitch_stats AS (
    SELECT
        pitch_type,
        COUNT(*)          AS pitches,
        {swing_count_sql()},
        {whiff_count_sql()}
    FROM pitches
    WHERE pitch_type IS NOT NULL
    GROUP BY pitch_type
)
SELECT
    pitch_type,
    pitches,
    ROUND(pitches * 100.0 / SUM(pitches) OVER (), 1) AS usage_pct,
    swings,
    ROUND(whiffs * 100.0 / NULLIF(swings, 0), 1)     AS whiff_pct
FROM pitch_stats
WHERE swings >= 100
ORDER BY whiff_pct DESC
"""
con.execute(q).df()

,pitch_type,pitches,usage_pct,swings,whiff_pct
0,FS,385,3.7,201.0,33.8
1,SL,1407,13.6,673.0,31.5
2,CH,1084,10.5,541.0,30.1
3,ST,666,6.4,302.0,26.8
4,CU,765,7.4,332.0,23.2
5,FC,909,8.8,436.0,18.8
6,FF,3541,34.2,1700.0,17.5
7,SI,1605,15.5,742.0,10.0


In [4]:
q = f"""
SELECT
    pitch_type,
    COUNT(*)                    AS pitches,
    COUNT(DISTINCT pitcher)     AS pitchers,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT pitcher), 1) AS per_pitcher
FROM pitches
WHERE pitch_type IS NOT NULL
GROUP BY pitch_type
ORDER BY pitchers DESC
"""
con.execute(q).df()

,pitch_type,pitches,pitchers,per_pitcher
0,FF,3541,199,17.8
1,SL,1407,140,10.1
2,SI,1605,139,11.5
3,CH,1084,118,9.2
4,FC,909,86,10.6
5,CU,765,78,9.8
6,ST,666,72,9.3
7,FS,385,36,10.7
8,KC,217,21,10.3
9,SV,45,7,6.4
